# Logits Preprocessing and Data Engineering

In [ ]:
def default_params(): 
    return {
        'current_model': 'M1', 
        'quantization': 'none', #['none',"int4", "int8", "float32", "float16"]
        'dataset': {
            'path': '/workspaces/CodeSmells/semeru-datasets/code_smells/prompts',
            'content_column': 'code',
            'sampling_size': 500,
            'prompt_id': 'P2', #['P1', 'P2', 'P3', 'P4'],
            'prompt_column': 'prompt',
        },
        'logging_path': '/workspaces/CodeSmells/datax/code_smells/logs', 
        'output_path' : '/workspaces/CodeSmells/datax/code_smells/logits/prompts',
        'callbacks_path' : '/workspaces/CodeSmells/datax/code_smells/callbacks/prompts',
        'cache_dir': '/workspaces/CodeSmells/datax/hugging_face_cache',
        'causal_models': {
            ##### BY ARCHITECTURE, SAME SIZE #####
            'M1' : 'codellama/CodeLlama-7b-hf', #https://huggingface.co/codellama/CodeLlama-7b-hf, 
            'M2' : 'mistralai/Mistral-7B-v0.3', #https://huggingface.co/mistralai/Mistral-7B-v0.3,
            'M3' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B,
            'M4' : 'bigcode/starcoder2-7b', #https://huggingface.co/bigcode/starcoder2-7b,
            ##### BY SIZE, SAME ARCHITECTURE #####
            'S1' : 'Qwen/Qwen2.5-Coder-0.5B', #https://huggingface.co/Qwen/Qwen2.5-Coder-0.5B,
            'S2' : 'Qwen/Qwen2.5-Coder-1.5B', #https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B,
            'S3' : 'Qwen/Qwen2.5-Coder-3B', #https://huggingface.co/Qwen/Qwen2.5-Coder-3B,
            'S4' : 'Qwen/Qwen2.5-Coder-7B', #https://huggingface.co/Qwen/Qwen2.5-Coder-7B,
        },
    }
params = default_params()


#### Imports

In [2]:
import pandas as pd
import os
import time
import numpy as np
import torch
import gc

In [3]:
import seaborn as sns
from scipy import stats
from statistics import NormalDist
import matplotlib.pyplot as plt

In [4]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

In [5]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [6]:
# Define log file path
log_file = f"{params['logging_path']}/{params['current_model']}"
create_folder(log_file)
log_file += '/data_en.txt'

# Create the log file if it doesn't exist
if not os.path.exists(log_file):
    with open(log_file, 'w'): 
        pass  # Create an empty log file

In [7]:
import logging
logging.basicConfig(filename=log_file, format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

#### Dataset

In [ ]:
print(f"{params['dataset']['path']}/{params['dataset']['prompt_id']}_{params['dataset']['sampling_size']}.json")

/workspaces/CodeSmells/semeru-datasets/code_smells/pipeline/curated_500.json


In [ ]:
df_dataset = pd.read_json(f"{params['dataset']['path']}/{params['dataset']['prompt_id']}_{params['dataset']['sampling_size']}.json", )

In [10]:
df_dataset

,id,commit_id,repo,path,file_name,fun_name,commit_message,code,url,language,...,n_ast_nodes,n_identifiers,s_msg_id,s_line,s_column,s_end_line,s_end_column,s_code,category,input_lenght
0,256261,a59bca366174d9c692fa19750c24d65f47660ef7,haystack,haystack/modeling/training/base.py,base.py,_get_state_dict,Apply black formatting (#2115)\n\n* Testing bl...,def _get_state_dict(self):\n \n ...,https://github.com/deepset-ai/haystack.git,Python,...,193,20,W0311,2,0,2,22,state_dict = {,Warning,296
1,305801,6f564e4f514b56bce281ec7e82703cfbff87b417,core,homeassistant/components/ring/binary_sensor.py,binary_sensor.py,async_added_to_hass,Improve entity type hints [r] (#77874),async def async_added_to_hass(self) -> None:\n...,https://github.com/home-assistant/core.git,Python,...,63,6,W0311,4,0,4,37,self._dings_update_callback(),Warning,73
2,70649,5fe901e5d86ed02dbbb63039a897582951266afd,wagtail,wagtail/admin/tests/pages/test_edit_page.py,test_edit_page.py,test_new_comment,Fix commenting thread notifications being sent...,def test_new_comment(self):\n post_data...,https://github.com/wagtail/wagtail.git,Python,...,565,37,C0301,33,0,33,125,self.assertEqual(mail.outbox[0].subjec...,Convention,666
3,151753,bdfedb5fcb02b88c600ef25c88bbb5d939b8bd0a,freqtrade,freqtrade/freqai/RL/BaseReinforcementLearningM...,BaseReinforcementLearningModel.py,__init__,Improve typehints / reduce warnings from mypy,"def __init__(self, **kwargs) -> None:\n ...",https://github.com/freqtrade/freqtrade.git,Python,...,411,38,W0311,13,0,13,44,import_str = 'stable_baselines3',Warning,494
4,3868,2282a4ae0221b1fb88e16eca8bc14a166998d2d2,airbyte,airbyte-integrations/connectors/source-hubspot...,streams.py,state,🎉 Source Hubspot: Migrate to CDK (#10177)\n\n*...,"def state(self, value):\n state_value =...",https://github.com/airbytehq/airbyte.git,Python,...,99,14,W0311,7,0,7,61,"self._start_date = max(self._state, se...",Warning,110
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1702546,43471,09f38ad3f6872bae5059a1de226362eb358c4a7a,airflow,tests/providers/microsoft/azure/operators/test...,test_asb.py,test_send_message_queue,Implement Azure Service Bus Queue Operators (#...,"def test_send_message_queue(self, mock_get_con...",https://github.com/apache/airflow.git,Python,...,132,21,C2801,10,12,13,24,mock.call()\n .__enter__()\n ...,Convention,193
1704801,196984,4a6d5d342e1d0111130d4b31708535b862bfacd0,sympy,sympy/printing/repr.py,repr.py,_print_Permutation,Update the deprecation for Permutation.print_c...,"def _print_Permutation(self, expr):\n f...",https://github.com/sympy/sympy.git,Python,...,388,30,C2801,20,16,20,53,Cycle(expr)(expr.size - 1).__repr__(),Convention,441
1705975,105,0b8a53bd313abdf484a9d1e3fbd6aad13c0ec857,PySyft,packages/syft/tests/syft/core/tensor/passthrou...,passthrough_test.py,test__rshift__,adding tests,def test__rshift__() -> None:\n data_a = np...,https://github.com/OpenMined/PySyft.git,Python,...,183,17,C2801,7,15,7,44,tensor_a.__rshift__(tensor_b),Convention,204
1717859,117464,9ce5a21dd6359fd7e8ebf78051ce9e97bd195ec9,mindsdb,tests/unit/executor_test_base.py,executor_test_base.py,set_executor,ML handler supbrocess (#3377)\n\n* log -> logg...,"def set_executor(self, to_mock_model_controlle...",https://github.com/mindsdb/mindsdb.git,Python,...,422,53,C2801,49,27,49,51,config_patch.__enter__(),Convention,610


#### Model Loading

In [ ]:
def instantiate_llm(model_name:str, cache_dir:str):
     '''Instantiate AutoModelForCausalLM'''
     tokenizer = AutoTokenizer.from_pretrained(model_name, cache_dir = cache_dir, use_fast=True)
     logging.info("Loaded AutoTokenizer - " + model_name)
     model = None
     if params['quantization'] == 'int4':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_4bit=True)
     elif params['quantization'] == 'int8':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, load_in_8bit=True)
     elif params['quantization'] == 'float32':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float32)
     elif params['quantization'] == 'float16':
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir, torch_dtype=torch.float16)
     else: 
          model = AutoModelForCausalLM.from_pretrained(model_name, cache_dir = cache_dir)
     logging.info("Loaded AutoModelForCausalLM - " + model_name)

     return tokenizer, model

In [12]:
tokenizer, model = instantiate_llm(params['causal_models'][params['current_model']], params['cache_dir'])

2025-05-22 15:27:49.348462: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1747927669.363448 2905793 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1747927669.368536 2905793 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-05-22 15:27:49.388025: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [13]:
print(model.config)
print(model.__class__)
print(tokenizer.__class__)

Starcoder2Config {
  "_attn_implementation_autoset": true,
  "activation_function": "gelu",
  "architectures": [
    "Starcoder2ForCausalLM"
  ],
  "attention_dropout": 0.1,
  "attention_softmax_in_fp32": true,
  "bos_token_id": 0,
  "embedding_dropout": 0.1,
  "eos_token_id": 0,
  "hidden_act": "gelu_pytorch_tanh",
  "hidden_size": 4608,
  "initializer_range": 0.018042,
  "intermediate_size": 18432,
  "layer_norm_epsilon": 1e-05,
  "max_position_embeddings": 16384,
  "mlp_type": "default",
  "model_type": "starcoder2",
  "norm_epsilon": 1e-05,
  "norm_type": "layer_norm",
  "num_attention_heads": 36,
  "num_hidden_layers": 32,
  "num_key_value_heads": 4,
  "residual_dropout": 0.1,
  "rope_scaling": null,
  "rope_theta": 1000000,
  "scale_attention_softmax_in_fp32": true,
  "scale_attn_weights": true,
  "sliding_window": 4096,
  "torch_dtype": "float32",
  "transformers_version": "4.51.3",
  "use_bias": true,
  "use_cache": true,
  "vocab_size": 49152
}

<class 'transformers.models.sta

#### preprocess dataset

In [ ]:
df_dataset['input_ids'] = df_dataset[params['dataset']['content_column']].map(lambda code: tokenizer.encode(code, add_special_tokens=False))
df_dataset['prompt_ids'] = df_dataset[params['dataset']['prompt_column']].map(lambda code: tokenizer.encode(code, add_special_tokens=False))
df_dataset['input_lenght'] = df_dataset['input_ids'].map(lambda input_ids: len(input_ids))

#### Softmax Normalization and Data Engineering

In [32]:
def topk_tuple(logit_vocab_tensor, largest, tokenizer_fn):
    """
    Return the decoded top-1 (or bottom-1) token and its logit value.
    """
    topk = logit_vocab_tensor.topk(k=1, largest=largest)
    top_token_id = topk.indices[0].item()
    decoded_token = tokenizer_fn.decode([top_token_id])  # Already handles special tokens and spaces
    return (decoded_token, topk.values[0].item())

In [33]:
def analyze_logits(logit_tensor_sequence, input_token_ids, tokenizer_fn, skip_first_token=True):
    """
    Analyze logits for each token in a sequence.

    Args:
        logit_tensor_sequence (List[Tensor]): List of vocab-sized logits for each token position.
        input_token_ids (List[int] or Tensor): Token IDs of the input prompt.
        tokenizer_fn: HuggingFace tokenizer with .decode() method.
        skip_first_token (bool): Whether to skip the first token prediction (default: True).

    Returns:
        dict with:
            - "max_cases": list of (decoded top-1 token, logit value)
            - "min_cases": list of (decoded bottom-1 token, logit value)
            - "actual_logits": list of (decoded ground-truth token, logit value)
    """
    max_cases = []
    min_cases = []
    actual_logits = []

    start_index = 1 if skip_first_token else 0
    token_targets = input_token_ids[start_index:]

    for position, token_id in enumerate(token_targets):
        vocab_logits = logit_tensor_sequence[position]

        # Top-1 max and min predictions
        max_case = topk_tuple(logit_vocab_tensor=vocab_logits, largest=True, tokenizer_fn=tokenizer_fn)
        min_case = topk_tuple(logit_vocab_tensor=vocab_logits, largest=False, tokenizer_fn=tokenizer_fn)

        # Actual token logit
        decoded_token = tokenizer_fn.decode([int(token_id)])
        logit_value = vocab_logits[int(token_id)].item()
        actual_case = (decoded_token, logit_value)

        max_cases.append(max_case)
        min_cases.append(min_case)
        actual_logits.append(actual_case)

    return {
        "max_cases": max_cases,
        "min_cases": min_cases,
        "actual_logits": actual_logits
    }

In [18]:
soft = torch.nn.Softmax( dim = 0 ) #Flattening normalization

In [ ]:
callbacks_dir = f"{params['callbacks_path']}/{params['dataset']['prompt_id']}/{params['current_model']}_q_{params['quantization']}"
out = np.load(f"{callbacks_dir}/logits_tensor[0]_batch[0].npy")

print(out.shape) #<sample,tokens,voc_tokens>
out = out[0]


(1, 250, 49152)


In [ ]:
input_ids_list = tokenizer.batch_encode_plus(df_dataset[params['dataset']['content_column']].tolist())
input_ids_list = [torch.tensor(  input_ids, dtype = torch.int) for input_ids in input_ids_list.input_ids]

logit_dict = analyze_logits(
    logit_tensor_sequence = [ soft( torch.from_numpy(token) ) for token in out] , #Out is a complete sequence
    input_token_ids = input_ids_list[0], ## SAMPLE ID
    tokenizer_fn = tokenizer
)

In [31]:
assert len(set(len(v) for v in logit_dict.values())) == 1, "All key array values in logit_dict do not have the same length"

#### Processing all the Batches

In [ ]:
def process_logit_batches(tokenizer, tokenized_inputs, num_samples=10000, skip_first_token=True):
    """
    Process multiple saved logits files and extract:
    - top-1 max logit predictions,
    - top-1 min logit predictions,
    - actual logits for ground-truth tokens.

    Args:
        tokenizer: HuggingFace tokenizer instance.
        tokenized_inputs (List[Tensor]): Tokenized input prompts (one per sample).
        num_samples (int): Number of samples to process.
        skip_first_token (bool): Whether to skip the first token prediction.

    Returns:
        Tuple of lists: (max_logit_predictions, min_logit_predictions, actual_logits)
    """
    max_logit_predictions = []
    min_logit_predictions = []
    actual_logit_scores = []

    softmax_fn = torch.nn.Softmax(dim=0)
    base_path = f"{params['callbacks_path']}/{params['dataset']['prompt_id']}/{params['current_model']}_q_{params['quantization']}"

    for sample_idx in range(num_samples):
        logits_file_path = f"{base_path}/logits_tensor[{sample_idx}]_batch[{sample_idx}].npy"
        logits_array = np.load(logits_file_path)[0]  # Shape: [sequence_length, vocab_size]

        # Apply softmax to each token’s logits
        normalized_logits = [softmax_fn(torch.from_numpy(token_logits)) for token_logits in logits_array]

        # Analyze logits using the unified function
        result = analyze_logits(
            logit_tensor_sequence=normalized_logits,
            input_token_ids=tokenized_inputs[sample_idx],
            tokenizer_fn=tokenizer,
            skip_first_token=skip_first_token
        )

        max_logit_predictions.append(result["max_cases"])
        min_logit_predictions.append(result["min_cases"])
        actual_logit_scores.append(result["actual_logits"])

        logging.info(f"Processed sample {sample_idx}")
        print(f"Processed sample {sample_idx}")

    return max_logit_predictions, min_logit_predictions, actual_logit_scores

In [22]:
input_ids_list = tokenizer.batch_encode_plus(df_dataset[params['dataset']['content_column']].tolist())
input_ids_list = [torch.tensor(  input_ids, dtype = torch.int) for input_ids in input_ids_list.input_ids]

In [ ]:
max_logit_token_prompt, min_logit_token_prompt, actual_logit_token_prompt = process_logit_batches(
    tokenizer=tokenizer , tokenized_inputs=input_ids_list, 
    num_samples = len(df_dataset)
) #<---WARNING TIME Consuming

#### Saving results

In [69]:
def create_folder(path):
    if not os.path.exists(path):
        os.makedirs(path)

In [70]:
dataframe_to_save = df_dataset.copy()
dataframe_to_save['max_prob'] = max_logit_token_prompt
dataframe_to_save['min_prob'] = min_logit_token_prompt
dataframe_to_save['actual_prob'] = actual_logit_token_prompt
dataframe_to_save.shape

(5, 33)

In [ ]:
output_dir = f"{params['output_path']}/{params['dataset']['prompt_id']}/{params['current_model']}_q_{params['quantization']}"
create_folder(output_dir)
dataframe_to_save.to_json(f"{output_dir}/raw_logits.json", index=False)

#### Loss Retrieval

In [ ]:
def batching_loss( size = dataframe_to_save.shape[0] ):
    output_dir = f"{params['callbacks_path']}/{params['dataset']['prompt_id']}/{params['current_model']}_q_{params['quantization']}"
    output_loss = []
    for current_batch in range(size):
        out = np.load(f"{output_dir}/_loss_batch[{current_batch}].npy")
        output_loss.append( out.item() ) #.item() for numpy library
        logging.info(current_batch)
    return output_loss

In [76]:
output_loss = batching_loss() #[WAENING!] Takes Time

In [77]:
output_loss

[0.6501871347427368,
 1.36393404006958,
 0.6265979409217834,
 0.9561377763748169,
 1.176944613456726]

In [78]:
dataframe_to_save['loss'] = output_loss
dataframe_to_save.head(5)

,id,commit_id,repo,path,file_name,fun_name,commit_message,code,url,language,...,s_end_line,s_end_column,s_code,category,input_lenght,input_ids,max_prob,min_prob,actual_prob,loss
0,256261,a59bca366174d9c692fa19750c24d65f47660ef7,haystack,haystack/modeling/training/base.py,base.py,_get_state_dict,Apply black formatting (#2115)\n\n* Testing bl...,def _get_state_dict(self):\n \n ...,https://github.com/deepset-ai/haystack.git,Python,...,2,22,state_dict = {,Warning,295,"[822, 903, 657, 29918, 3859, 29918, 8977, 2989...","[(<PRE>, 0.7492886185646057), (module, 0.47944...","[(<s>, 6.321602312800434e-13), ($}, 1.21012630...","[(def, 0.0007611338514834642), (_, 0.009010307...",0.650187
1,305801,6f564e4f514b56bce281ec7e82703cfbff87b417,core,homeassistant/components/ring/binary_sensor.py,binary_sensor.py,async_added_to_hass,Improve entity type hints [r] (#77874),async def async_added_to_hass(self) -> None:\n...,https://github.com/home-assistant/core.git,Python,...,4,37,self._dings_update_callback(),Warning,72,"[7465, 822, 7465, 29918, 23959, 29918, 517, 29...","[(<PRE>, 0.7492926716804504), (\n, 0.145589932...","[(<s>, 6.322408417115677e-13), (oreferrer, 3.7...","[(async, 1.0811768333951477e-05), (def, 0.0006...",1.363934
2,70649,5fe901e5d86ed02dbbb63039a897582951266afd,wagtail,wagtail/admin/tests/pages/test_edit_page.py,test_edit_page.py,test_new_comment,Fix commenting thread notifications being sent...,def test_new_comment(self):\n post_data...,https://github.com/wagtail/wagtail.git,Python,...,33,125,self.assertEqual(mail.outbox[0].subjec...,Convention,665,"[822, 1243, 29918, 1482, 29918, 9342, 29898, 1...","[(<PRE>, 0.7492899298667908), (module, 0.47944...","[(<s>, 6.321468413832132e-13), ($}, 1.21012033...","[(def, 0.0007611322216689587), (test, 0.019638...",0.626598
3,151753,bdfedb5fcb02b88c600ef25c88bbb5d939b8bd0a,freqtrade,freqtrade/freqai/RL/BaseReinforcementLearningM...,BaseReinforcementLearningModel.py,__init__,Improve typehints / reduce warnings from mypy,"def __init__(self, **kwargs) -> None:\n ...",https://github.com/freqtrade/freqtrade.git,Python,...,13,44,import_str = 'stable_baselines3',Warning,493,"[822, 4770, 2344, 12035, 1311, 29892, 3579, 19...","[(<PRE>, 0.7492900490760803), (module, 0.47944...","[(<s>, 6.321457571810407e-13), ($}, 1.21011436...","[(def, 0.0007611316395923495), (__, 0.00598208...",0.956138
4,3868,2282a4ae0221b1fb88e16eca8bc14a166998d2d2,airbyte,airbyte-integrations/connectors/source-hubspot...,streams.py,state,🎉 Source Hubspot: Migrate to CDK (#10177)\n\n*...,"def state(self, value):\n state_value =...",https://github.com/airbytehq/airbyte.git,Python,...,7,61,"self._start_date = max(self._state, se...",Warning,109,"[822, 2106, 29898, 1311, 29892, 995, 1125, 13,...","[(<PRE>, 0.749289870262146), (module, 0.479445...","[(<s>, 6.321889626376143e-13), ($}, 1.21010756...","[(def, 0.0007611394976265728), (state, 6.87381...",1.176945


In [79]:
## Saving CheckPoint 2
dataframe_to_save.to_json(f"{output_dir}/raw_logits.json", index=False)

In [80]:
print("================================= PROCESS COMPLETED =================================")

================================= PROCESS COMPLETE =================================


In [82]:
del model
torch.cuda.empty_cache()
gc.collect()

0